# 04 · Evaluate and reload: select, test once, prove the artefact, and the capstone

*Base Model Training*, notebook 4 of 4

A model is ready for its final evaluation only when every development decision is fixed. In this notebook you choose a checkpoint with validation data alone, open the test rows exactly once, look beyond a single accuracy number, save the chosen model as a self-contained artefact, and prove that a freshly rebuilt model reproduces its predictions. The capstone at the end asks you to run a complete, reproducible experiment of your own, with one controlled change, and to defend its conclusion.

## What you will build

- `snapshot(params)`: a copy of the weights that later updates cannot change;
- `confusion_matrix(y_true, y_pred, n_classes)` and `recall_and_precision(cm)`: errors class by class;
- `most_confident_mistakes(probs, y_true, k)`: the errors worth reading first;
- `load_for_inference(path)`: a working model rebuilt from a file alone;
- the capstone: a protocol written in advance, one controlled change across seeds, one test evaluation, and a report.

## Learning objectives

By the end of this notebook you will be able to:

1. select a checkpoint with validation data, and keep a real copy of it;
2. evaluate the selected model once on held-out test data and compare it with a baseline;
3. read a confusion matrix, per-class recall and precision, and individual mistakes;
4. save a model with its configuration and preprocessing, rebuild it, and verify its predictions within a tolerance;
5. report variation across seeds without choosing the best seed after the fact;
6. plan, run and document a small, reproducible base-model experiment.

### How to work through this notebook

Run the cells from top to bottom. At each **✍️ Your turn** you type a function yourself, then run the check cell beneath it:

- ✅ means your function matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not match yet. In both cases the notebook carries on with a hidden reference version, so *Run All* always works and you can come back later.

Each reference solution sits in a collapsed cell just above the function you type. Open it only when you are stuck.

## Dependencies

- **Tier 1:** NumPy for every calculation and Matplotlib for one plot of the learning curves; the numbers behind the plot are also printed. The standard-library modules `json`, `os`, `sys` and `tempfile` write the model's configuration, build file paths, record the Python version and provide a scratch folder. Nothing else is imported, so the notebook runs in JupyterLite in your browser, on the course JupyterHub server and in Google Colab.
- The saved artefact goes to a temporary folder, so the notebook leaves nothing behind in your workspace.

## Setup

Run this cell first. It imports the libraries, creates a seeded random generator and defines `practice`, the helper that checks each function you write against its reference.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import json
import os
import sys
import tempfile

import matplotlib.pyplot as plt

As before, `practice_via` checks a function through a **probe**, a small function that runs your code on a fresh example and returns plain numbers to compare.

In [ ]:
def practice_via(name, yours, reference, probe, cases, atol=1e-6):
    """Like practice, but compares probe(function, *case) for your function and for the reference.

    Use it when a function works by changing objects (filling in .grad, updating
    weights) or when several answers are correct: the probe runs the function on
    a fresh example and turns the outcome into plain numbers. Returns your
    function once the measurements match, otherwise the reference.
    """
    def measure_yours(*case):
        return probe(yours, *case)

    def measure_reference(*case):
        return probe(reference, *case)

    chosen = practice(name, measure_yours, measure_reference, cases, atol)
    return yours if chosen is measure_yours else reference

## 📦 Recap: what you built earlier

The hidden cell below defines the tiny tensor engine from notebook 1; the data pieces from notebook 2 (`make_readings`, `group_split`, `fit_standardiser` and `standardise`); and the training pieces from notebook 3 (`Linear`, `dropout`, `MLP`, `adam_update`, `Adam` and `train_step`). Run it; open it only if you want to look back at the code.

In [ ]:
# ── Tiny tensor autograd (NumPy) ── the engine you built in Base Model Training ──
import numpy as np


def _unbroadcast(grad, shape):
    """Sum a gradient back down to `shape`, undoing NumPy broadcasting."""
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for axis, size in enumerate(shape):
        if size == 1 and grad.shape[axis] != 1:
            grad = grad.sum(axis=axis, keepdims=True)
    return grad


class Tensor:
    """A NumPy array that remembers how it was made, so gradients can flow back."""

    def __init__(self, data, requires_grad=False, parents=(), backward=None, op=""):
        self.data = np.asarray(data, dtype=np.float64)
        self.requires_grad = requires_grad
        self.grad = None
        self._parents = parents
        self._backward = backward  # grad_out -> one gradient per parent
        self.op = op

    shape = property(lambda self: self.data.shape)
    ndim = property(lambda self: self.data.ndim)
    T = property(lambda self: self.transpose())

    def __repr__(self):
        return f"Tensor(shape={self.shape}, op={self.op or 'leaf'}, requires_grad={self.requires_grad})"

    @staticmethod
    def lift(value):
        return value if isinstance(value, Tensor) else Tensor(value)

    def _result(self, data, parents, backward, op):
        if any(parent.requires_grad for parent in parents):
            return Tensor(data, True, parents, backward, op)
        return Tensor(data, op=op)

    # arithmetic, with broadcasting
    def __add__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data + other.data, (self, other), lambda g: (g, g), "add")

    __radd__ = __add__

    def __neg__(self):
        return self._result(-self.data, (self,), lambda g: (-g,), "neg")

    def __sub__(self, other):
        return self + (-Tensor.lift(other))

    def __rsub__(self, other):
        return Tensor.lift(other) + (-self)

    def __mul__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data * other.data, (self, other),
                            lambda g: (g * other.data, g * self.data), "mul")

    __rmul__ = __mul__

    def __truediv__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data / other.data, (self, other),
                            lambda g: (g / other.data, -g * self.data / other.data ** 2), "div")

    def __rtruediv__(self, other):
        return Tensor.lift(other) / self

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float)), "only constant exponents"
        return self._result(self.data ** exponent, (self,),
                            lambda g: (g * exponent * self.data ** (exponent - 1),), "pow")

    def __matmul__(self, other):
        other = Tensor.lift(other)

        def backward(g):
            return (g @ np.swapaxes(other.data, -1, -2), np.swapaxes(self.data, -1, -2) @ g)

        return self._result(self.data @ other.data, (self, other), backward, "matmul")

    # elementwise functions
    def exp(self):
        out = np.exp(self.data)
        return self._result(out, (self,), lambda g: (g * out,), "exp")

    def log(self):
        return self._result(np.log(self.data), (self,), lambda g: (g / self.data,), "log")

    def tanh(self):
        out = np.tanh(self.data)
        return self._result(out, (self,), lambda g: (g * (1 - out ** 2),), "tanh")

    def relu(self):
        return self._result(np.maximum(self.data, 0), (self,), lambda g: (g * (self.data > 0),), "relu")

    def sigmoid(self):
        out = 0.5 * (1 + np.tanh(0.5 * self.data))  # stable for large |x|
        return self._result(out, (self,), lambda g: (g * out * (1 - out),), "sigmoid")

    # reductions and reshaping
    def sum(self, axis=None, keepdims=False):
        def backward(g):
            if axis is not None and not keepdims:
                g = np.expand_dims(g, axis)
            return (np.broadcast_to(g, self.shape).copy(),)

        return self._result(self.data.sum(axis=axis, keepdims=keepdims), (self,), backward, "sum")

    def mean(self, axis=None, keepdims=False):
        count = self.data.size if axis is None else np.prod([self.shape[a] for a in np.atleast_1d(axis)])
        return self.sum(axis=axis, keepdims=keepdims) / count

    def reshape(self, *shape):
        return self._result(self.data.reshape(*shape), (self,), lambda g: (g.reshape(self.shape),), "reshape")

    def transpose(self, *axes):
        axes = axes or tuple(reversed(range(self.ndim)))
        inverse = np.argsort(axes)
        return self._result(self.data.transpose(axes), (self,), lambda g: (g.transpose(inverse),), "transpose")

    def __getitem__(self, index):
        def backward(g):
            full = np.zeros_like(self.data)
            np.add.at(full, index, g)  # repeated indices (an embedding row used twice) add up
            return (full,)

        return self._result(self.data[index], (self,), backward, "index")

    def log_softmax(self, axis=-1):
        shifted = self.data - self.data.max(axis=axis, keepdims=True)
        out = shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))
        probs = np.exp(out)
        return self._result(out, (self,), lambda g: (g - probs * g.sum(axis=axis, keepdims=True),), "log_softmax")

    def softmax(self, axis=-1):
        shifted = np.exp(self.data - self.data.max(axis=axis, keepdims=True))
        out = shifted / shifted.sum(axis=axis, keepdims=True)
        return self._result(out, (self,), lambda g: (out * (g - (g * out).sum(axis=axis, keepdims=True)),), "softmax")

    # reverse-mode differentiation
    def backward(self, grad=None):
        order, seen = [], set()

        def visit(node):
            if id(node) not in seen:
                seen.add(id(node))
                for parent in node._parents:
                    visit(parent)
                order.append(node)

        visit(self)
        self.grad = np.ones_like(self.data) if grad is None else np.asarray(grad, dtype=np.float64)
        for node in reversed(order):
            if node._backward is None or node.grad is None:
                continue
            for parent, parent_grad in zip(node._parents, node._backward(node.grad)):
                if not parent.requires_grad or parent_grad is None:
                    continue
                parent_grad = _unbroadcast(np.asarray(parent_grad, dtype=np.float64), parent.shape)
                parent.grad = parent_grad if parent.grad is None else parent.grad + parent_grad


def parameter(*shape, rng, scale=None):
    """A trainable weight, initialised with N(0, scale^2); default scale is He's sqrt(2 / fan_in)."""
    scale = np.sqrt(2.0 / shape[0]) if scale is None else scale
    return Tensor(rng.normal(0.0, scale, shape), requires_grad=True)


def zeros_parameter(*shape):
    return Tensor(np.zeros(shape), requires_grad=True)


def zero_grad(params):
    for p in params:
        p.grad = None


def cross_entropy(logits, targets, mask=None):
    """Mean next-token/class cross-entropy. logits: (..., classes); targets: int (...); mask: 1.0 keeps a position."""
    flat = logits.reshape(-1, logits.shape[-1]).log_softmax(axis=-1)
    targets = np.asarray(targets).reshape(-1)
    picked = flat[np.arange(targets.size), targets]
    if mask is None:
        return -picked.mean()
    weights = np.asarray(mask, dtype=np.float64).reshape(-1)
    return -(picked * weights).sum() / max(weights.sum(), 1.0)


def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    centred = x - mean
    var = (centred ** 2).mean(axis=-1, keepdims=True)
    return centred / (var + eps) ** 0.5 * gamma + beta


def numerical_grad(f, tensor, eps=1e-6):
    """Central-difference gradient of scalar f() with respect to tensor.data."""
    grad = np.zeros_like(tensor.data)
    it = np.nditer(tensor.data, flags=["multi_index"])
    for _ in it:
        index = it.multi_index
        original = tensor.data[index]
        tensor.data[index] = original + eps
        plus = float(f().data)
        tensor.data[index] = original - eps
        minus = float(f().data)
        tensor.data[index] = original
        grad[index] = (plus - minus) / (2 * eps)
    return grad


def relative_error(a, b):
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

# ── From notebook 2: generated data, split by source, standardiser ─────────────
def make_readings(n_sources, rows_per_source, rng, spread=0.3, label_noise=0.1, flipped_fraction=0.0):
    """Generated sensor readings. Returns X (rows, 2) raw measurements, y (rows,) labels in {0, 1}
    and groups (rows,) source ids; the rows of one source cluster around that source's centre."""
    centres = rng.normal(size=(n_sources, 2))
    flipped = rng.random(n_sources) < flipped_fraction
    groups = np.repeat(np.arange(n_sources), rows_per_source)
    signal = centres[groups] + spread * rng.normal(size=(groups.size, 2))
    noisy = rng.random(groups.size) < label_noise
    y = ((signal[:, 0] * signal[:, 1] > 0) ^ flipped[groups] ^ noisy).astype(int)
    X = np.column_stack([50.0 + 10.0 * signal[:, 0], 0.5 + 0.1 * signal[:, 1]])
    return X, y, groups


def group_split(groups, fractions, rng):
    sources = np.unique(groups)
    shuffled = rng.permutation(sources)
    n_train = round(fractions[0] * len(sources))
    n_val = round(fractions[1] * len(sources))
    parts = shuffled[:n_train], shuffled[n_train:n_train + n_val], shuffled[n_train + n_val:]
    return tuple(np.flatnonzero(np.isin(groups, part)) for part in parts)


def fit_standardiser(X):
    return X.mean(axis=0), np.maximum(X.std(axis=0), 1e-6)


def standardise(X, mean, scale):
    return (X - mean) / scale

# ── From notebook 3: modules, Adam and the training step ──────────────────────
class Linear:
    """A dense layer: x @ W + b, with W of shape (n_in, n_out) and b of shape (n_out,)."""

    def __init__(self, n_in, n_out, rng):
        self.W = parameter(n_in, n_out, rng=rng)
        self.b = zeros_parameter(n_out)

    def __call__(self, x):
        return x @ self.W + self.b

    def parameters(self):
        return [self.W, self.b]


def dropout(x, p, training, rng):
    if not training or p == 0:
        return x
    keep = rng.random(x.shape) >= p
    return x * (keep / (1.0 - p))


class MLP:
    """Dense layers with ReLU between them, and dropout after each hidden ReLU while training."""

    def __init__(self, sizes, rng, dropout_p=0.0):
        self.sizes, self.dropout_p = list(sizes), dropout_p
        self.layers = [Linear(n_in, n_out, rng) for n_in, n_out in zip(sizes[:-1], sizes[1:])]
        self.training = True

    def __call__(self, x, rng=None):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                x = dropout(x.relu(), self.dropout_p, self.training, rng)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

    def train(self):
        self.training = True

    def eval(self):
        self.training = False


def adam_update(param, grad, m, v, t, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    m = beta1 * m + (1 - beta1) * grad
    v = beta2 * v + (1 - beta2) * grad ** 2
    m_hat = m / (1 - beta1 ** t)
    v_hat = v / (1 - beta2 ** t)
    return param - lr * m_hat / (np.sqrt(v_hat) + eps), m, v


class Adam:
    """Adam: per-parameter step sizes from running averages of the gradient and its square."""

    def __init__(self, params, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params, self.lr, self.beta1, self.beta2, self.eps = list(params), lr, beta1, beta2, eps
        self.m = [np.zeros_like(p.data) for p in self.params]
        self.v = [np.zeros_like(p.data) for p in self.params]
        self.t = 0

    def zero_grad(self):
        zero_grad(self.params)

    def step(self):
        self.t += 1
        for i, p in enumerate(self.params):
            if p.grad is None:
                continue
            new_value, self.m[i], self.v[i] = adam_update(p.data, p.grad, self.m[i], self.v[i], self.t,
                                                          self.lr, self.beta1, self.beta2, self.eps)
            p.data[...] = new_value


def train_step(model, optimiser, X, y, rng):
    model.train()
    optimiser.zero_grad()
    loss = cross_entropy(model(Tensor(X), rng), y)
    if not np.isfinite(loss.data):
        raise FloatingPointError(f"non-finite training loss ({float(loss.data)}): check the learning rate and the data")
    loss.backward()
    optimiser.step()
    return float(loss.data)

The next cell rebuilds the data and the split from notebook 2. It then hands the test rows to a `SealedTestSet`, an object that gives them out exactly once, and blanks them out of the development arrays: their inputs become `nan` and their labels become `-1`. If any development step touched a test row by accident, the `nan` or the invalid label would show up immediately.

In [ ]:
class SealedTestSet:
    """Hands out the test rows exactly once, so that they cannot quietly steer development decisions."""

    def __init__(self, X, y):
        self._X, self._y, self.times_opened = X.copy(), y.copy(), 0

    def open(self):
        if self.times_opened:
            raise RuntimeError("This test set has already been used once. A decision made after seeing "
                               "test results needs fresh test data for an independent evaluation.")
        self.times_opened += 1
        return self._X, self._y


X, y, groups = make_readings(160, 5, np.random.default_rng(0))
train_rows, val_rows, test_rows = group_split(groups, (0.6, 0.2, 0.2), np.random.default_rng(1))
sealed_test = SealedTestSet(X[test_rows], y[test_rows])
mean, scale = fit_standardiser(X[train_rows])
Z_dev, y_dev = standardise(X, mean, scale), y.copy()
Z_dev[test_rows], y_dev[test_rows] = np.nan, -1          # development code cannot see the test rows
dev_data = (Z_dev, y_dev, train_rows, val_rows)
majority_class = int(np.bincount(y_dev[train_rows]).argmax())
print(f"{train_rows.size} training and {val_rows.size} validation rows for development; "
      f"{test_rows.size} test rows sealed; majority training class: {majority_class}")

## 1 · Select a checkpoint with validation data, and keep a real copy

A long run passes through many sets of weights. **Checkpoint selection** keeps the one with the best validation score, here the lowest validation loss, because training loss keeps falling long after the model has started to memorise noise. That is a development decision, like the choice of learning rate, so it uses validation data only.

There is a trap in *keeping* the best weights. The optimisers update `p.data` **in place**, so a list such as `[p.data for p in params]` holds references to the very arrays that the next steps overwrite: the "best" weights silently follow every later update, and you end up with the last weights. A real checkpoint needs **copies**.

In [ ]:
weights = np.zeros(3)
kept_reference, kept_copy = weights, weights.copy()
weights -= 0.5                     # an in-place update, like optimiser.step()
print("kept by reference:", kept_reference, "| kept as a copy:", kept_copy)

### ✍️ Your turn: `snapshot`

Write `snapshot(params)`:

- **input:** a list of tensors, such as `model.parameters()`;
- **output:** a list of NumPy arrays, independent copies of each tensor's `data`, in the same order.

The probe takes a snapshot, then changes every parameter in place, and returns the snapshot: a copy keeps the old values, while a list of references would show the new ones.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def snapshot_reference(params):
    return [p.data.copy() for p in params]

In [ ]:
def snapshot(params):
    """Independent copies of every parameter's values, in order."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
def snapshot_probe(snapshot_fn, seed):
    """Probe: snapshot fresh parameters, change them in place, and return the snapshot."""
    params = MLP([2, 4, 2], np.random.default_rng(seed)).parameters()
    kept = snapshot_fn(params)
    for p in params:
        p.data += 1.0                  # later "training" changes the parameters in place
    return kept


snapshot = practice_via("snapshot", snapshot, snapshot_reference, snapshot_probe, cases=[(0,), (1,)])

**Expected behaviour.** ✅ when your function copies. If you return `[p.data for p in params]`, the check shows values that are all larger by exactly 1: the snapshot followed the change. The next cell uses `snapshot` in a training loop that evaluates on the validation rows every 50 steps, keeps the best weights seen so far, and restores them at the end.

In [ ]:
def evaluate_rows(model, Z, y, rows):
    """Evaluation mode, one forward pass, no update. Returns (mean loss, accuracy)."""
    model.eval()
    logits = model(Tensor(Z[rows]))
    return float(cross_entropy(logits, y[rows]).data), float((logits.data.argmax(axis=1) == y[rows]).mean())


def train_and_select(config, data, seed, steps=1000, eval_every=50, batch_size=32):
    """Train an MLP with Adam and keep the checkpoint with the lowest validation loss.

    config: {"sizes", "dropout_p", "learning_rate"}; data: (Z, y, train_rows, val_rows).
    Returns a dict with the model (holding the selected weights), the selected step,
    its validation loss and the history of (step, training loss, validation loss).
    """
    Z, y, train_rows, val_rows = data
    model = MLP(config["sizes"], np.random.default_rng(seed), config["dropout_p"])
    optimiser = Adam(model.parameters(), lr=config["learning_rate"])
    rng = np.random.default_rng(seed + 1000)
    best = {"val_loss": np.inf, "step": 0, "weights": snapshot(model.parameters())}
    history = []
    for step in range(1, steps + 1):
        rows = train_rows[rng.choice(train_rows.size, batch_size, replace=False)]
        train_step(model, optimiser, Z[rows], y[rows], rng)
        if step % eval_every == 0:
            train_loss, _ = evaluate_rows(model, Z, y, train_rows)
            val_loss, _ = evaluate_rows(model, Z, y, val_rows)
            history.append((step, train_loss, val_loss))
            if val_loss < best["val_loss"]:
                best = {"val_loss": val_loss, "step": step, "weights": snapshot(model.parameters())}
    for p, kept in zip(model.parameters(), best["weights"]):
        p.data[...] = kept                       # restore the selected checkpoint
    model.eval()
    return {"model": model, "best_step": best["step"], "best_val_loss": best["val_loss"], "history": history}


config = {"sizes": [2, 64, 64, 2], "dropout_p": 0.1, "learning_rate": 0.01}
run = train_and_select(config, dev_data, seed=0)
selected = run["model"]
for step, train_loss, val_loss in run["history"][1::2]:
    print(f"step {step:4}: training loss {train_loss:.3f}, validation loss {val_loss:.3f}")
print(f"selected step {run['best_step']} with validation loss {run['best_val_loss']:.3f}; "
      f"the last step had {run['history'][-1][2]:.3f}")

In [ ]:
history = np.array(run["history"])
fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.plot(history[:, 0], history[:, 1], marker=".", label="training loss")
ax.plot(history[:, 0], history[:, 2], marker=".", label="validation loss")
ax.axvline(run["best_step"], linestyle="--", color="grey", label=f"selected: step {run['best_step']}")
ax.set_xlabel("optimiser step")
ax.set_ylabel("cross-entropy")
ax.set_title("Checkpoint selection on validation loss")
ax.legend()
plt.show()

**What to notice.** Both losses drop steeply during the first hundred steps. After that the training loss keeps creeping down, while the validation loss wanders within a band, roughly 0.29 to 0.36 here, as each mini-batch nudges the decision boundary back and forth. The selected checkpoint is the lowest point of that band, and the last checkpoint is a little worse. Two cautions apply. The minimum of a noisy curve is partly luck, so the selected validation loss is slightly optimistic, which is one more reason to keep the test rows for the end. And when a network can memorise its training rows, the validation loss rises steadily after its minimum instead of wandering, which makes selection matter even more.

In [ ]:
val_loss_selected, val_accuracy_selected = evaluate_rows(selected, Z_dev, y_dev, val_rows)
baseline_val_accuracy = float((y_dev[val_rows] == majority_class).mean())
print(f"selected checkpoint on validation: loss {val_loss_selected:.3f}, accuracy {val_accuracy_selected:.1%}; "
      f"majority baseline {baseline_val_accuracy:.1%}")

## 2 · Test once, after every decision is made

Architecture, learning rate, training length and checkpoint are now fixed. Only now do the test rows enter, once, to measure the decision. If the test result made you change anything and test again, the test set would have become development feedback, and its second score would no longer be an independent estimate; a new decision would need fresh test data. Keep the original baseline, split definition and number of test rows in the report, because a percentage means little without them.

The next cell opens the sealed test set. Run it once: running it a second time raises an error on purpose. If you need to start again, run the notebook from the top.

In [ ]:
X_test_raw, y_test = sealed_test.open()                 # the only time the test rows are read
selected.eval()
test_logits = selected(Tensor(standardise(X_test_raw, mean, scale)))
test_probs = test_logits.softmax(axis=-1).data
test_pred = test_probs.argmax(axis=1)
test_accuracy = float((test_pred == y_test).mean())
baseline_test_accuracy = float((y_test == majority_class).mean())
print(f"test accuracy {test_accuracy:.1%} on {y_test.size} rows from {np.unique(groups[test_rows]).size} sources; "
      f"majority-class baseline {baseline_test_accuracy:.1%}")

### Look beyond one number

A single accuracy hides *which* errors the model makes. A **confusion matrix** counts, for every true class (row) and every predicted class (column), how many examples landed there; the diagonal holds the correct predictions. From it:

- the **recall** of class $k$ is the fraction of true-$k$ examples that were predicted as $k$, the diagonal entry divided by its row sum;
- the **precision** of class $k$ is the fraction of $k$ predictions that were right, the diagonal entry divided by its column sum.

In [ ]:
tiny_true, tiny_pred = np.array([0, 0, 1, 1, 1]), np.array([0, 1, 1, 1, 0])
print("true:", tiny_true, "| predicted:", tiny_pred, "| pairs:", list(zip(tiny_true.tolist(), tiny_pred.tolist())))

### ✍️ Your turn: `confusion_matrix`

Write `confusion_matrix(y_true, y_pred, n_classes)`:

- **inputs:** integer arrays `y_true` and `y_pred` of the same length; `n_classes`;
- **output:** an integer array of shape `(n_classes, n_classes)` whose entry `[t, p]` counts the examples with true class `t` and predicted class `p`.

Hint: start from `np.zeros((n_classes, n_classes), dtype=int)` and add 1 for each pair; `np.add.at(cm, (y_true, y_pred), 1)` does it in one line.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def confusion_matrix_reference(y_true, y_pred, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=int)
    np.add.at(cm, (y_true, y_pred), 1)
    return cm

In [ ]:
def confusion_matrix(y_true, y_pred, n_classes):
    """Counts of (true class, predicted class) pairs, shape (n_classes, n_classes)."""
    # ✍️ Your code here (two or three lines)
    raise NotImplementedError

In [ ]:
confusion_matrix = practice("confusion_matrix", confusion_matrix, confusion_matrix_reference, cases=[
    (tiny_true, tiny_pred, 2),
    (np.array([2, 0, 2, 2, 0, 1]), np.array([0, 0, 2, 2, 0, 2]), 3),
    (np.array([1, 1]), np.array([1, 1]), 3),                    # edge case: classes that never occur
])

### ✍️ Your turn: `recall_and_precision`

Write `recall_and_precision(cm)`:

- **input:** a confusion matrix with true classes in rows;
- **output:** a tuple `(recall, precision)` of float arrays, one entry per class: the diagonal divided by the row sums, and the diagonal divided by the column sums. Where a sum is 0 the rate is undefined; return 0.0 there.

Hint: `np.divide(a, b, out=np.zeros(len(a)), where=b > 0)` divides only where `b > 0`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def recall_and_precision_reference(cm):
    diagonal = np.diag(cm).astype(float)
    rows, columns = cm.sum(axis=1), cm.sum(axis=0)
    recall = np.divide(diagonal, rows, out=np.zeros(len(diagonal)), where=rows > 0)
    precision = np.divide(diagonal, columns, out=np.zeros(len(diagonal)), where=columns > 0)
    return recall, precision

In [ ]:
def recall_and_precision(cm):
    """Per-class recall (diagonal / row sums) and precision (diagonal / column sums); 0.0 where undefined."""
    # ✍️ Your code here (about four lines)
    raise NotImplementedError

In [ ]:
recall_and_precision = practice("recall_and_precision", recall_and_precision, recall_and_precision_reference, cases=[
    (np.array([[1, 1], [1, 2]]),),
    (np.array([[2, 0, 0], [0, 0, 1], [1, 0, 2]]),),
    (np.array([[0, 0, 0], [0, 2, 0], [0, 0, 0]]),),             # edge case: empty rows and columns
])
test_cm = confusion_matrix(y_test, test_pred, 2)
test_recall, test_precision = recall_and_precision(test_cm)
print("test confusion matrix (rows: true class, columns: predicted class):\n", test_cm)
for k in range(2):
    print(f"class {k}: support {test_cm[k].sum():3}, recall {test_recall[k]:.1%}, precision {test_precision[k]:.1%}")

**Expected behaviour.** For the five-example demo the matrix is `[[1, 1], [1, 2]]`: recall is 50% for class 0 and 67% for class 1. On the test rows both classes have similar support, and recall and precision lie between about 82% and 88% for each class: the errors are not concentrated in one class. With imbalanced classes, this is where a model that ignores the rare class would be exposed despite a high accuracy.

### Read the mistakes themselves

Aggregates say how many mistakes there are; individual examples suggest *why*. The most useful ones to read first are the **confident** mistakes, where the model put high probability on the wrong class. Because these data are generated, we can undo the measurement units, $s_1 = (x_1 - 50)/10$ and $s_2 = (x_2 - 0.5)/0.1$, and check each mistake against the true rule, "class 1 when $s_1 s_2 > 0$". With real data you would look at the raw inputs and, ideally, at the original records.

### ✍️ Your turn: `most_confident_mistakes`

Write `most_confident_mistakes(probs, y_true, k)`:

- **inputs:** `probs`, predicted probabilities of shape `(n, classes)`; `y_true`, integer labels; `k`, how many to return;
- **output:** an integer array with the row indices of at most `k` misclassified examples, ordered from the highest to the lowest probability assigned to the (wrong) predicted class.

Hints: `pred = probs.argmax(axis=1)`; `wrong = np.flatnonzero(pred != y_true)`; `confidence = probs[wrong, pred[wrong]]`; sort with `np.argsort(-confidence, kind="stable")` so that ties keep their order, and keep the first `k`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def most_confident_mistakes_reference(probs, y_true, k):
    pred = probs.argmax(axis=1)
    wrong = np.flatnonzero(pred != y_true)
    confidence = probs[wrong, pred[wrong]]
    return wrong[np.argsort(-confidence, kind="stable")[:k]]

In [ ]:
def most_confident_mistakes(probs, y_true, k):
    """Indices of up to k misclassified rows, most confident first."""
    # ✍️ Your code here (about four lines)
    raise NotImplementedError

In [ ]:
demo_probs = np.array([[0.9, 0.1], [0.4, 0.6], [0.2, 0.8], [0.7, 0.3], [0.55, 0.45]])
demo_labels = np.array([1, 1, 0, 0, 1])
most_confident_mistakes = practice("most_confident_mistakes", most_confident_mistakes, most_confident_mistakes_reference, cases=[
    (demo_probs, demo_labels, 2),
    (demo_probs, demo_labels, 10),                  # fewer mistakes than k
    (demo_probs, demo_probs.argmax(axis=1), 3),     # edge case: no mistakes at all
])

signal_test = np.column_stack([(X_test_raw[:, 0] - 50.0) / 10.0, (X_test_raw[:, 1] - 0.5) / 0.1])
rule_label = (signal_test[:, 0] * signal_test[:, 1] > 0).astype(int)
mistakes = most_confident_mistakes(test_probs, y_test, test_pred.size)
print(f"{mistakes.size} mistakes; most confident first:")
for i in list(mistakes[:3]) + list(mistakes[-3:]):
    print(f"  row {i:3}: signal ({signal_test[i, 0]:+.2f}, {signal_test[i, 1]:+.2f}), label {y_test[i]}, "
          f"rule says {rule_label[i]}, predicted {test_pred[i]} with probability {test_probs[i, test_pred[i]]:.2f}")
disagree = (rule_label[mistakes] != y_test[mistakes]).mean()
print(f"labels that contradict the rule among all mistakes: {disagree:.0%}")

**Expected behaviour.** For the demo, rows 0 and 2 are the confident mistakes (0.9 and 0.8), then rows 1 and 4. On the test rows, the most confident mistakes sit deep inside a quadrant, and their labels contradict the rule: they are flipped labels, so the model is right and the data are wrong. The least confident mistakes sit close to an axis, near the decision boundary, where the model is genuinely uncertain. About three quarters of all the mistakes have labels that contradict the rule. Two failure groups with two different remedies, better labels and more data near the boundary, were invisible in the single accuracy number.

## 3 · Save an artefact, rebuild it, and prove it

An **inference artefact** must let someone rebuild the model without your notebook: the architecture's configuration (layer sizes, dropout rate), the weights, the preprocessing (the training mean and scale), the meaning of each output (class names, in order), and the versions of the software that produced it. A file that loads without error can still be wrong, because it may use the wrong class order or forget the standardiser, so verify it: record the logits for a fixed set of raw inputs *before* saving, rebuild a fresh model from the file alone, and compare the logits within a small tolerance. This round trip checks the artefact path; it is not another evaluation of quality.

In [ ]:
def save_for_inference(path, model, mean, scale, class_names):
    """Everything needed to rebuild and use the model: configuration, weights, preprocessing and versions."""
    config = {"sizes": model.sizes, "dropout_p": model.dropout_p, "class_names": list(class_names),
              "numpy_version": np.__version__, "python_version": sys.version.split()[0]}
    arrays = {f"param_{i}": p.data for i, p in enumerate(model.parameters())}
    np.savez(path, config=json.dumps(config), mean=mean, scale=scale, **arrays)


artefact_dir = tempfile.mkdtemp(prefix="base_models_")
artefact_path = os.path.join(artefact_dir, "selected_model.npz")
reference_inputs = X[val_rows[::16]]                 # ten fixed raw inputs, before any preprocessing
selected.eval()
logits_before = selected(Tensor(standardise(reference_inputs, mean, scale))).data
save_for_inference(artefact_path, selected, mean, scale, ["class 0", "class 1"])
with np.load(artefact_path) as saved:
    print(f"saved {os.path.getsize(artefact_path):,} bytes:", saved.files)

### ✍️ Your turn: `load_for_inference`

Write `load_for_inference(path)`:

- **input:** the path of a file written by `save_for_inference`;
- **output:** a tuple `(model, mean, scale, class_names)`. Rebuild the model with `MLP(config["sizes"], np.random.default_rng(0), config["dropout_p"])`, copy every `param_i` into it in place, put it into **evaluation mode**, and return copies of the saved `mean` and `scale` (the file closes when the `with` block ends).

Hints: `with np.load(path) as saved:` and `config = json.loads(str(saved["config"]))`. The random generator only fills the weights before they are overwritten, so its seed does not matter.

The probe loads the file with your function, reports whether the model is in evaluation mode, the class names, and the logits for the ten reference inputs.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def load_for_inference_reference(path):
    with np.load(path) as saved:
        config = json.loads(str(saved["config"]))
        model = MLP(config["sizes"], np.random.default_rng(0), config["dropout_p"])
        for i, p in enumerate(model.parameters()):
            p.data[...] = saved[f"param_{i}"]
        mean, scale = saved["mean"].copy(), saved["scale"].copy()
    model.eval()
    return model, mean, scale, config["class_names"]

In [ ]:
def load_for_inference(path):
    """Rebuild (model in evaluation mode, mean, scale, class_names) from a saved artefact."""
    # ✍️ Your code here (about eight lines)
    raise NotImplementedError

In [ ]:
def inference_probe(load_fn, path, raw_inputs):
    """Probe: load with load_fn and report (evaluation mode?, class names, logits for raw_inputs)."""
    model, mean_, scale_, names = load_fn(path)
    if model.training:
        return (False, names, "no logits: the model is still in training mode")
    return (True, names, model(Tensor(standardise(raw_inputs, mean_, scale_))).data)


load_for_inference = practice_via("load_for_inference", load_for_inference, load_for_inference_reference,
                                  inference_probe, cases=[(artefact_path, reference_inputs)])

reloaded, mean_loaded, scale_loaded, class_names = load_for_inference(artefact_path)
logits_after = reloaded(Tensor(standardise(reference_inputs, mean_loaded, scale_loaded))).data
round_trip_gap = float(np.abs(logits_after - logits_before).max())
print(f"largest logit difference after the round trip: {round_trip_gap:.1e}; class names {class_names}")
assert np.allclose(logits_after, logits_before, rtol=0.0, atol=1e-12), "the reloaded model must reproduce the logits"

**Expected behaviour.** ✅, and a largest difference of `0.0e+00`: `.npz` files store `float64` numbers exactly, and the same operations run in the same order. A tolerance (here $10^{-12}$) is still the right comparison, because on another machine or library version the last digits may legitimately differ.

## 4 · Variation across seeds

The random seed decides the initial weights, the batches and the dropout masks, so a second run of the same configuration lands somewhere slightly different. Reporting that spread is honest. Picking the best of several seeds *after* seeing their results, and reporting that one, is not: the maximum of several noisy numbers is biased upwards. The cell repeats the selected configuration with two more seeds and reports the validation results only, because the test set has already been used for this configuration.

In [ ]:
seed_runs = [run] + [train_and_select(config, dev_data, seed=seed) for seed in (1, 2)]
seed_accuracies = [evaluate_rows(r["model"], Z_dev, y_dev, val_rows)[1] for r in seed_runs]
seed_losses = [r["best_val_loss"] for r in seed_runs]
for seed, r, accuracy in zip((0, 1, 2), seed_runs, seed_accuracies):
    print(f"seed {seed}: selected step {r['best_step']:4}, validation loss {r['best_val_loss']:.3f}, accuracy {accuracy:.1%}")
print(f"validation accuracy {np.mean(seed_accuracies):.1%} ± {np.std(seed_accuracies):.1%} over 3 seeds "
      f"(the best seed alone would claim {max(seed_accuracies):.1%})")

**What to notice.** The three seeds select different steps and land a point or two apart in validation accuracy. That spread is the noise floor for any comparison: a change that improves validation accuracy by less than this should not be called an improvement on the evidence of one run each.

## 5 · 🧪 Experiments

### 🧪 Experiment 1: keep the best weights by reference

The cell takes a "best so far" snapshot, trains for 50 more steps with the in-place Adam optimiser, and compares the kept weights with the current ones. Switch `keep_by` between the two options.

In [ ]:
keep_by = "reference"  # 🧪 try "copy"

probe_model = MLP([2, 16, 2], np.random.default_rng(50))
probe_optimiser, probe_rng = Adam(probe_model.parameters(), lr=0.01), np.random.default_rng(51)
kept = [p.data for p in probe_model.parameters()] if keep_by == "reference" else snapshot(probe_model.parameters())
for _ in range(50):
    rows = train_rows[probe_rng.choice(train_rows.size, 32, replace=False)]
    train_step(probe_model, probe_optimiser, Z_dev[rows], y_dev[rows], probe_rng)
drift = max(float(np.abs(k - p.data).max()) for k, p in zip(kept, probe_model.parameters()))
print(f"kept by {keep_by}: largest difference between the kept and the current weights after 50 steps = {drift:.4f}")
print("The kept weights silently followed every update." if drift == 0 else "The kept weights stayed put: a real checkpoint.")

**What to notice.** Kept by reference, the difference is exactly 0: the "best" checkpoint *is* the current model. Nothing raises an error, so this bug can survive for a long time. It would report the last epoch while claiming to report the best one.

### 🧪 Experiment 2: a reload that goes wrong quietly

Each option below rebuilds the artefact correctly but uses it wrongly. Compare the logits and the predicted class names with the originals.

In [ ]:
mistake = "no standardiser"  # 🧪 try "training mode", "reversed class names" or "none"

model_r, mean_r, scale_r, names_r = load_for_inference(artefact_path)
inputs_r = standardise(reference_inputs, mean_r, scale_r)
if mistake == "no standardiser":
    inputs_r = reference_inputs                          # raw units straight into the network
if mistake == "training mode":
    model_r.train()                                      # dropout stays on
if mistake == "reversed class names":
    names_r = names_r[::-1]
logits_r = model_r(Tensor(inputs_r), np.random.default_rng(0)).data
named_before = [class_names[k] for k in logits_before.argmax(axis=1)]
named_after = [names_r[k] for k in logits_r.argmax(axis=1)]
print(f"mistake = {mistake!r}: largest logit difference {np.abs(logits_r - logits_before).max():.3g}; "
      f"{sum(a != b for a, b in zip(named_before, named_after))} of 10 predicted names changed")

**What to notice.** Without the standardiser, raw values around 50 switch off every unit of the second hidden layer, so the network returns the same logits, its final bias, for every input, and half of the ten predictions flip. In training mode, dropout makes the logits random, here up to about 0.5 away from the originals; none of these ten predictions changes, but inputs near the decision boundary would. Reversed class names leave the logits identical, yet every predicted *name* is wrong. The round-trip check in section 3 catches the first two mistakes because it compares logits on fixed inputs instead of merely checking that the file loads; storing the class names, in order, inside the artefact prevents the third.

## 6 · Put it together: the evaluation protocol, checked

The cell below asserts every property of the protocol you have just run: selection used validation data and kept the true best weights; the test rows were opened exactly once; the reported numbers are consistent with each other; and the artefact reproduces the selected model.

In [ ]:
history = np.array(run["history"])
assert run["best_step"] == int(history[np.argmin(history[:, 2]), 0]), "the selected step has the lowest validation loss"
assert np.isclose(evaluate_rows(selected, Z_dev, y_dev, val_rows)[0], run["best_val_loss"], rtol=0.0, atol=1e-12), \
    "the restored weights reproduce the selected validation loss"
assert run["best_val_loss"] <= history[-1, 2], "the selected checkpoint is at least as good as the last one on validation"
assert val_accuracy_selected >= baseline_val_accuracy + 0.2, "the selected model beats the majority baseline on validation"
assert sealed_test.times_opened == 1, "the test rows were read exactly once"
assert test_cm.sum() == y_test.size and np.trace(test_cm) == int((test_pred == y_test).sum()), "consistent confusion matrix"
assert test_accuracy >= 0.75, "test accuracy is far above the majority baseline of about 47%"
assert not reloaded.training and round_trip_gap <= 1e-12, "the artefact reproduces the selected model in evaluation mode"
print(f"Protocol checked: step {run['best_step']} selected on validation, test accuracy {test_accuracy:.1%} "
      f"(baseline {baseline_test_accuracy:.1%}, {y_test.size} rows), artefact verified.")

**Expected behaviour.** Every assertion passes. On this generated task, with 10% flipped labels, the selected model's test accuracy is about 85%, against a majority baseline below 50%; with only 160 test rows from 32 sources, a difference of a few points between two models would not be meaningful. None of this says anything about real sensors or any other population: the number describes one generated distribution.

## Exercises and capstone

### Exercise 1: `f1_scores`

The F1 score of a class combines its precision $P$ and recall $R$ into one number, their harmonic mean $2PR/(P + R)$, which is low when either is low. Write `f1_scores(cm)`, returning one F1 score per class and 0.0 where $P + R = 0$. You may call `recall_and_precision`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def f1_scores_reference(cm):
    recall, precision = recall_and_precision_reference(cm)
    total = recall + precision
    return np.divide(2 * recall * precision, total, out=np.zeros(len(total)), where=total > 0)

In [ ]:
def f1_scores(cm):
    """Per-class F1: 2PR / (P + R), with 0.0 where P + R = 0."""
    # ✍️ Your code here (about three lines)
    raise NotImplementedError

In [ ]:
f1_scores = practice("f1_scores", f1_scores, f1_scores_reference, cases=[
    (np.array([[1, 1], [1, 2]]),),
    (np.array([[2, 0, 0], [0, 0, 1], [1, 0, 2]]),),          # class 1 is never predicted correctly
    (np.zeros((2, 2), dtype=int),),                          # edge case: an empty matrix
])
print("F1 on the test rows:", f1_scores(test_cm).round(3))

### Exercise 2, the capstone: a reproducible base-model experiment

The capstone asks for a small but complete experiment that someone else could reproduce and question. The scaffold below runs from start to finish with default choices on a **new** task, so that it has its own, unused test set: three classes set by the distance of a hidden signal from the centre (an inner disc, a middle ring and an outer ring), generated with sources as before, and with 5% of the labels replaced at random.

Work through it in order:

1. **Write the protocol first.** Edit the `protocol` cell *before* you look at any result: the split unit, the baseline, the primary validation metric, the budget, the selection rule, the single controlled change and the seeds. Writing it down first is what stops the result from choosing the method.
2. **Run the comparison on validation data only.** The base configuration and the changed one are each trained with three seeds; nothing touches the test rows.
3. **Decide, then test once.** The configuration with the lower mean validation loss wins, and its seed-0 checkpoint becomes the artefact. Only then is the sealed test set opened.
4. **Save and verify the artefact.** Save the chosen model with its preprocessing and class names, rebuild it from the file, and compare logits on fixed inputs.
5. **Report.** Fill in the report prompts at the end in your own words.

In [ ]:
def make_rings(n_sources, rows_per_source, rng, spread=0.25, label_noise=0.05):
    """Generated three-class data: the class is set by the distance of a hidden signal from the centre
    (inner disc, middle ring, outer ring). Returns X (rows, 2) raw measurements, y (rows,) labels
    in {0, 1, 2} and groups (rows,) source ids."""
    centres = 1.2 * rng.normal(size=(n_sources, 2))
    groups = np.repeat(np.arange(n_sources), rows_per_source)
    signal = centres[groups] + spread * rng.normal(size=(groups.size, 2))
    labels = np.digitize(np.sqrt((signal ** 2).sum(axis=1)), [1.0, 1.8])
    replaced = rng.random(groups.size) < label_noise
    labels = np.where(replaced, rng.integers(0, 3, groups.size), labels)
    X = np.column_stack([3.0 * signal[:, 0] - 4.0, 0.5 * signal[:, 1] + 2.0])
    return X, labels, groups

In [ ]:
protocol = {
    "task": "three classes by the distance of a hidden signal from the centre; generated, 120 sources of 6 rows",
    "split": "by source: 60% training, 20% validation, 20% test; split seed 31",
    "baseline": "always predict the most common training class",
    "primary_metric": "validation cross-entropy of each run's selected checkpoint, averaged over the seeds",
    "budget": "800 Adam steps of 32 rows per run; validation every 50 steps",
    "selection": "within a run, the lowest validation loss; between configurations, the lower mean over seeds",
    "base_config": {"sizes": [2, 64, 64, 3], "dropout_p": 0.0, "learning_rate": 0.01},
    "controlled_change": {"sizes": [2, 16, 16, 3]},   # 🧪 one change only, such as {"dropout_p": 0.2} or {"learning_rate": 0.003}
    "seeds": [0, 1, 2],
    "artefact": "the seed-0 run of the winning configuration",
}
for key, value in protocol.items():
    print(f"{key:>17}: {value}")

In [ ]:
X_rings, y_rings, groups_rings = make_rings(120, 6, np.random.default_rng(30))
train_r, val_r, test_r = group_split(groups_rings, (0.6, 0.2, 0.2), np.random.default_rng(31))
sealed_rings_test = SealedTestSet(X_rings[test_r], y_rings[test_r])
mean_rings, scale_rings = fit_standardiser(X_rings[train_r])
Z_rings, y_rings_dev = standardise(X_rings, mean_rings, scale_rings), y_rings.copy()
Z_rings[test_r], y_rings_dev[test_r] = np.nan, -1
rings_data = (Z_rings, y_rings_dev, train_r, val_r)
majority_rings = int(np.bincount(y_rings_dev[train_r], minlength=3).argmax())
print(f"{train_r.size} training, {val_r.size} validation and {test_r.size} sealed test rows; "
      f"training class counts {np.bincount(y_rings_dev[train_r], minlength=3)}; "
      f"majority baseline on validation {(y_rings_dev[val_r] == majority_rings).mean():.1%}")

configurations = {"base": protocol["base_config"],
                  "changed": {**protocol["base_config"], **protocol["controlled_change"]}}
capstone = {}
for name, cfg in configurations.items():
    runs = [train_and_select(cfg, rings_data, seed, steps=800) for seed in protocol["seeds"]]
    losses = [r["best_val_loss"] for r in runs]
    accuracies = [evaluate_rows(r["model"], Z_rings, y_rings_dev, val_r)[1] for r in runs]
    capstone[name] = {"config": cfg, "runs": runs, "val_losses": losses, "val_accuracies": accuracies}
    print(f"{name:8} {cfg}: validation loss {np.mean(losses):.3f} ± {np.std(losses):.3f}, "
          f"accuracy {np.mean(accuracies):.1%} ± {np.std(accuracies):.1%}")

In [ ]:
winner = min(capstone, key=lambda name: np.mean(capstone[name]["val_losses"]))
gap = abs(np.mean(capstone["base"]["val_losses"]) - np.mean(capstone["changed"]["val_losses"]))
spread = max(np.std(capstone["base"]["val_losses"]), np.std(capstone["changed"]["val_losses"]))
artefact = capstone[winner]["runs"][0]["model"]           # decided in the protocol: the seed-0 run

X_rings_test, y_rings_test = sealed_rings_test.open()      # once, after the decision
rings_pred = artefact(Tensor(standardise(X_rings_test, mean_rings, scale_rings))).data.argmax(axis=1)
rings_cm = confusion_matrix(y_rings_test, rings_pred, 3)
rings_recall, rings_precision = recall_and_precision(rings_cm)
report = {
    "protocol": protocol,
    "validation": {name: {"mean_loss": round(float(np.mean(c["val_losses"])), 4),
                          "std_loss": round(float(np.std(c["val_losses"])), 4),
                          "mean_accuracy": round(float(np.mean(c["val_accuracies"])), 4)} for name, c in capstone.items()},
    "decision": winner,
    "difference_vs_seed_spread": [round(float(gap), 4), round(float(spread), 4)],
    "test": {"rows": int(y_rings_test.size), "accuracy": round(float((rings_pred == y_rings_test).mean()), 4),
             "majority_baseline": round(float((y_rings_test == majority_rings).mean()), 4),
             "confusion_matrix": rings_cm.tolist(), "recall": rings_recall.round(3).tolist()},
    "versions": {"numpy": np.__version__, "python": sys.version.split()[0]},
}
for key in ("validation", "decision", "difference_vs_seed_spread", "test"):
    print(f"{key}: {json.dumps(report[key])}")
assert sealed_rings_test.times_opened == 1 and report["test"]["accuracy"] > report["test"]["majority_baseline"]

In [ ]:
rings_probs = artefact(Tensor(standardise(X_rings_test, mean_rings, scale_rings))).softmax(axis=-1).data
rings_path = os.path.join(artefact_dir, "capstone_model.npz")
fixed_inputs = X_rings[val_r[::12]]                         # fixed raw validation rows
logits_saved = artefact(Tensor(standardise(fixed_inputs, mean_rings, scale_rings))).data
save_for_inference(rings_path, artefact, mean_rings, scale_rings, ["inner disc", "middle ring", "outer ring"])
rebuilt, mean_rebuilt, scale_rebuilt, names_rebuilt = load_for_inference(rings_path)
reload_gap = float(np.abs(rebuilt(Tensor(standardise(fixed_inputs, mean_rebuilt, scale_rebuilt))).data - logits_saved).max())
report["reload"] = {"verified": reload_gap <= 1e-12, "largest_logit_gap": reload_gap, "class_names": names_rebuilt}
print("reload:", json.dumps(report["reload"]))
print("three most confident test mistakes (rows of the test set):", most_confident_mistakes(rings_probs, y_rings_test, 3))
assert report["reload"]["verified"] and not rebuilt.training, "the rebuilt capstone model must reproduce its logits"

**What to notice in the default run.** Both configurations reach similar validation losses, and the difference between their means is far smaller than the spread across seeds. An honest conclusion is therefore modest: "under this protocol, the narrower network performed about as well as the wider one; the difference is within seed-to-seed variation". The test result then describes only the chosen artefact. Change `controlled_change` (one change only), run the capstone cells again from the protocol cell down, and compare.

### Your report

Answer each prompt in two to four sentences, in a markdown cell of your own, using the numbers that *your* run printed:

1. **Protocol.** What was decided before any result was seen, and why is the split by source?
2. **Result.** What did the controlled change do to the mean validation loss and accuracy, compared with the spread across seeds? Was it an improvement, a failure, or no detectable difference?
3. **Test.** What was the test accuracy of the artefact, against the majority baseline and with the number of test rows? Why was the test set opened only after the decision?
4. **Three mistakes.** Use `most_confident_mistakes(rings_probs, y_rings_test, 3)` and look at those rows of `X_rings_test`, `y_rings_test` and `rings_probs`. Describe three errors: are they replaced labels, boundary cases, or something else?
5. **Limitations.** What does this experiment *not* show? Consider the generated data, the sample size and any population beyond it.
6. **Reproduction.** Which seeds, versions and cells recreate your numbers? Restart the kernel, run every cell again, and confirm that the report is identical.

### Exercise 3: an open question

Suppose your controlled change improves the mean validation accuracy by 1 point, while the three seeds of each configuration spread over 2 points. What would you do before claiming an improvement, and what would change your mind?

## Common errors and debugging

- **Selecting the checkpoint, or the configuration, on the test set.** Selection is a development decision; use validation data, then test once.
- **Testing again after changing something.** The second test score is no longer independent; treat the first test set as development data and obtain fresh test data.
- **The "best" checkpoint equals the last one.** It was kept by reference while the optimiser updated in place; copy the arrays.
- **Reloaded predictions differ from the originals.** The preprocessing was not saved or not applied, the model is still in training mode, or the configuration does not match the weights.
- **Correct logits but wrong labels.** The class order was not saved with the model; store the class names, in order, in the artefact.
- **Reporting the best of several seeds.** Report the mean and the spread, decided in advance; the maximum is biased upwards.
- **`RuntimeError: This test set has already been used once`.** A cell that opens the sealed test set ran twice. That is the point of the guard; re-run the notebook from the top if you need a clean start.

## Summary

- Choose the checkpoint, like every other development decision, with validation data, and keep **copies** of the best weights, because optimisers update in place.
- Open the test set once, after the decisions are fixed, and report it with the baseline, the number of rows and the split definition.
- Confusion matrices, per-class recall and precision, and individual confident mistakes reveal failure groups that one accuracy number hides.
- An inference artefact holds configuration, weights, preprocessing, class names and versions; verify it by comparing logits on fixed inputs after a fresh rebuild.
- Report the spread across seeds, decide the protocol before the comparison, and phrase conclusions no more strongly than the evidence allows.

## Next

You have built a tensor engine, a data pipeline, a training loop with diagnostics and exact checkpoints, and an evaluation protocol, and you have trained and defended a base model with them. The *Intermediate Model Training* course improves a working model through controlled experiments: regularisation, class imbalance, transfer learning, efficient optimisation and reliable model selection.